## Introduction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# df = pd.read_csv('./UPL25_matches2.csv')
df = pd.read_csv('./FWSL25_matches2.csv')

In [ ]:
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns#all numeric columns
num_cols = ['duration','distance_km','sprint_distance_m','power_plays','energy_kcal','impacts',
                   'player_load','top_speed_kmh','distance_per_min_mmin','power_score_wkg','work_ratio','max_acceleration_mss','max_deceleration_mss']# numeric columns of high interest
intensity_metrics =['player_load','top_speed_kmh','distance_per_min_mmin','power_score_wkg','work_ratio','max_acceleration_mss','max_deceleration_mss']
volume_metrics = ['distance_km','sprint_distance_m','power_plays','energy_kcal','impacts']

In [ ]:
from IPython.display import HTML

def style_table_for_docs(df, hide_index=False):
    styled = (
        df.style
        .set_properties(**{
            'color': 'black',
            'background-color': 'white',
            'border': '1px solid black',
            'text-align': 'center',
            'padding': '2px'  # reduces row height
        })
        .set_table_styles([
            {'selector': 'th', 'props': [
                ('color', 'black'), 
                ('border', '1px solid black'),
                ('padding', '2px')  # also reduce header row height
            ]},
            {'selector': 'td', 'props': [
                ('color', 'black'), 
                ('border', '1px solid black'),
                ('padding', '2px')
            ]},
            {'selector': 'table', 'props': [
                ('border', '2px solid black'), 
                ('border-collapse', 'collapse')
            ]}
        ])
    )
    
    if hide_index:
        styled = styled.hide(axis='index')

    return styled

In [ ]:
df['total_accelerations'] = df[
	[
		'accelerations_zone_count:_1__2_mss',
		'accelerations_zone_count:_2__3_mss',
		'accelerations_zone_count:_3__4_mss',
		'accelerations_zone_count:_>_4_mss'
	]
].sum(axis=1)

df['total_decelerations'] = df[
	[
		'deceleration_zone_count:_1__2_mss',
		'deceleration_zone_count:_2__3_mss',
		'deceleration_zone_count:_3__4_mss',
		'deceleration_zone_count:_>_4_mss'
	]
].sum(axis=1)
df['total_actions'] = df['total_accelerations'] + df['total_decelerations']

In [ ]:
volume_metrics.append('total_accelerations')
volume_metrics.append('total_decelerations')
volume_metrics.append('total_actions')

df['acc_counts_per_min'] = df['total_accelerations'] / df['duration']
df['dec_counts_per_min'] = df['total_decelerations'] / df['duration']

intensity_metrics.append('acc_counts_per_min')
intensity_metrics.append('dec_counts_per_min')

In [ ]:
uploaded_matches = {
    "She Maroons FC": 20,
    "Kawempe Muslim LFC": 22,
    "Uganda Martyrs Lubaga WFC": 18,
    "Rines SS WFC": 22,
    "Amus College WFC": 19,
    "Wakiso Hill WFC": 18,
    "Lady Doves FC": 16,
    "Makerere University WFC": 22,
    "Kampala Queens FC": 17,
    "Olila HS WFC": 13,
    "She Corporates FC": 5,
    "FC Tooro Queens": 0,   
}

required_total_uploaded_matches = 22

In [ ]:
matchday_order = [f'Md{i}' for i in range(1, 23)]

In [ ]:
def plot_line_with_values(data, x_col, y_col, title=None, x_label=None, y_label=None, figsize=(10, 6)):
    plt.figure(figsize=figsize)
    sns.lineplot(data=data, x=x_col, y=y_col, marker='o', color='orange', linewidth=2)
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    gray_color = '#808080'
    ax.spines['bottom'].set_color(gray_color)
    ax.spines['left'].set_color(gray_color)

    if x_col == 'Match Day':
        data[x_col] = pd.Categorical(data[x_col], categories=matchday_order, ordered=True)
        data_sorted = data.sort_values(by=x_col)
        xticklabels = [str(md).replace('Md', 'MD ') for md in data_sorted[x_col]]
        ax.set_xticks(range(len(data_sorted[x_col])))
        ax.set_xticklabels(xticklabels, rotation=90, fontsize=10)
        sns.lineplot(data=data_sorted, x=x_col, y=y_col, marker='o', color='orange', linewidth=2)

    for i, (x, y) in enumerate(zip(data[x_col], data[y_col])):
        ax.annotate(
            f'{y}',
            (x, y),
            textcoords="offset points",
            xytext=(0, 10),
            ha='center',
            fontsize=8,
            bbox=dict(boxstyle="circle,pad=0.35", edgecolor='#FE912A', facecolor='white', linewidth=1)
        )

    if title is None:
        title = f'{y_col} vs {x_col}'
    plt.title(title)
    if x_label is None:
        x_label = x_col.replace('_', ' ').title()
    plt.xlabel(x_label)
    if y_label is None:
        y_label = y_col.replace('_', ' ').title()
    plt.ylabel(y_label)
    plt.grid(axis='y', linestyle='--', linewidth=0.5, color='#E0DCDD', alpha=0.7)
    plt.tight_layout()
    return ax


## Club Specifics


In [ ]:
for i in uploaded_matches:
    print(i)

In [ ]:
club = 'She Maroons FC'
club = club.title() 
club_df = df[df['club_for'] == club].copy()

In [ ]:
club_df['match_day'] = club_df['match_day'].str.replace(r'^Wmd(\d+)$', r'Md\1', regex=True) # adjustment for women clubs

In [ ]:
def get_matchday_stats_styled(club_df, matchday_order, style_table_for_docs, club):
  # 1. Aggregate per match_day
  match_stats = (
    club_df
      .groupby('match_day', as_index=False)
      .agg({
        'club_against':'first',  # opponent club
        'p_name': 'nunique',     # unique players
        'duration': 'mean',       # average session duration
        'result':'first',
        'location': 'first',
      })
      .rename(columns={
        'club_against':'Opponent Club',
        'match_day': 'Match Day',
        'p_name': 'Number of Players Monitored',
        'duration': 'Average Session Duration (min)',
        'result': 'Match Result',
        'location': 'Match Location',
      })
  )

  # 2. Ensure all match days are present, fill missing with NaN/0
  full_md_df = pd.DataFrame({'Match Day': matchday_order})
  match_stats = full_md_df.merge(match_stats, on='Match Day', how='left')

  # Fill text columns with NaN and numeric columns with 0 for missing match days
  text_cols = ['Opponent Club', 'Match Result', 'Match Location']
  num_cols = ['Number of Players Monitored', 'Average Session Duration (min)']

  for col in text_cols:
    match_stats[col] = match_stats[col].where(match_stats[col].notna(), np.nan)
  for col in num_cols:
    match_stats[col] = match_stats[col].fillna(0)

  # 3. Optionally round the duration column
  match_stats['Average Session Duration (min)'] = match_stats['Average Session Duration (min)'].round(1)

  # 4. Style for docs
  styled = style_table_for_docs(match_stats)
  styled = styled.set_caption(f"Matches Monitored for {club}")
  return styled,match_stats


In [ ]:
def plot_players_per_matchday(club_df, matchday_order, style_table_for_docs, club):
    match_stats = get_matchday_stats_styled(club_df, matchday_order, style_table_for_docs, club)[1]
    ax1 = plot_line_with_values(
        match_stats,
        'Match Day',
        'Number of Players Monitored',
        title='Player Number per Matchday',
        x_label='Match Day',
        y_label='Number of Players'
    )
    return ax1

In [ ]:
def get_players_monitored_stats(club_df, style_table_for_docs, club):
    # 1. Aggregate to get position (first seen) and count of unique match days per player
    player_stats = (
        club_df
          .groupby('p_name', as_index=False)
          .agg({
              'general_position': 'first',
              'match_day': pd.Series.nunique
          })
          .rename(columns={'match_day': 'Match Days Analysed'})
    )

    # 2. Title‑case player names and sort
    player_stats['p_name'] = player_stats['p_name'].str.title()
    player_stats = player_stats.sort_values('p_name')

    # 3. Rename columns for display
    player_stats = player_stats.rename(columns={
        'p_name': 'Player Name',
        'general_position': 'Position'
    })

    # 4. Display with your styling function
    styled = style_table_for_docs(player_stats, hide_index=True)
    styled = styled.set_caption(f"Players Monitored for {club}")
    return styled, player_stats


In [ ]:
def get_metric_summary_styled(club_df, volume_metrics, intensity_metrics, club, style_table_for_docs):
    x= club_df[volume_metrics + intensity_metrics].agg(['sum','max','min','mean','std'])
    x.rename(columns={
                                            'distance_km':'Distance',
                                            'sprint_distance_m':'Sprint Distance',
                                            'power_plays':'Power Plays',
                                            'energy_kcal':'Energy',
                                            'impacts':'Impacts',
                                            'total_accelerations':'Accelerations',
                                            'total_decelerations': 'Decelerations',
                                            'total_actions':'Total Actions',
                                            'player_load':'Player Load',
                                            'top_speed_kmh':'Top Speed',
                                            'distance_per_min_mmin':'Distance per Minute',
                                            'power_score_wkg':'Power Score',
                                            'work_ratio':'Work Ratio',
                                            'max_acceleration_mss':'Max Acceleration',
                                            'max_deceleration_mss':'Max Deceleration',
                                            'acc_counts_per_min':'Acceleration Counts per Min',
                                            'dec_counts_per_min':'Deceleration Counts per Min'},inplace=True)
    x = x.T.reset_index().rename(columns={'index': 'Metric','sum': 'Total', 'max': 'Max', 'min': 'Min', 'mean': 'Mean', 'std': 'Std Dev'})[['Metric', 'Total', 'Max', 'Min', 'Mean', 'Std Dev']]
    x['Range'] = x['Max'] - x['Min']
    
    return style_table_for_docs(x.round(2)).set_caption(f"Metric Summary for {club}"), x.round(2)

In [ ]:
def get_top_players_by_metric(club_df, metrics, style_table_for_docs):
    top_players = []
    for metric in metrics:
        idx = club_df[metric].idxmax()
        row = club_df.loc[idx]
        top_players.append({
            'metric': metric,
            'player': row['p_name'],
            'value': row[metric],
            'match day': row['match_day']
        })
    top_players_df = pd.DataFrame(top_players)
    top_players_df['value'] = top_players_df['value'].round(2)
    top_players_df = pd.DataFrame(top_players).set_index('metric').T


    top_players_df.rename(columns={'distance_km':'Distance','sprint_distance_m':'Sprint Distance','power_plays':'Power Plays','energy_kcal':'Energy',
                                            'impacts':'Impacts','total_accelerations':'Accelerations',
                                            'total_decelerations': 'Decelerations','total_actions':'Total Actions',
                                            'player_load':'Player Load','top_speed_kmh':'Top Speed',
                                            'distance_per_min_mmin':'Distance per Minute','power_score_wkg':'Power Score','work_ratio':'Work Ratio',
                                            'max_acceleration_mss':'Max Acceleration','max_deceleration_mss':'Max Deceleration',
                                            'acc_counts_per_min':'Acceleration Counts per Min','dec_counts_per_min':'Deceleration Counts per Min'},inplace=True)
    top_players_df.drop(columns=['Max Acceleration','Max Deceleration'], inplace=True) 
    top_players_df = top_players_df.T
    top_players_df.reset_index(inplace=True)
    styled = style_table_for_docs(top_players_df.T)

    return styled, top_players_df

In [ ]:
def get_avg_metrics_by_position(club_df, volume_metrics, intensity_metrics, style_table_for_docs):

    avg_metrics_by_position = club_df.groupby('general_position')[volume_metrics + intensity_metrics].mean().round(2).reset_index()

    avg_metrics_by_position.rename(columns={'general_position': 'Position',
                                            'distance_km':'Distance',
                                            'sprint_distance_m':'Sprint Distance',
                                            'power_plays':'Power Plays',
                                            'energy_kcal':'Energy',
                                            'impacts':'Impacts',
                                            'total_accelerations':'Accelerations',
                                            'total_decelerations': 'Decelerations',
                                            'total_actions':'Total Actions',
                                            'player_load':'Player Load',
                                            'top_speed_kmh':'Top Speed',
                                            'distance_per_min_mmin':'Distance per Minute',
                                            'power_score_wkg':'Power Score',
                                            'work_ratio':'Work Ratio',
                                            'max_acceleration_mss':'Max Acceleration',
                                            'max_deceleration_mss':'Max Deceleration',
                                            'acc_counts_per_min':'Acceleration Counts per Min',
                                            'dec_counts_per_min':'Deceleration Counts per Min'},inplace=True)
    
    avg_metrics_by_position.drop(columns=['Max Acceleration','Max Deceleration'], inplace=True) 
    styled = style_table_for_docs(avg_metrics_by_position)
    return styled, avg_metrics_by_position.T.reset_index()

In [ ]:
def get_avg_metric_per_matchday(club_df, volume_metrics, intensity_metrics, matchday_order,style_table_for_docs):
    avg_metric_per_matchday = club_df.groupby('match_day')[volume_metrics + intensity_metrics].mean().reset_index()
    avg_metric_per_matchday['match_day'] = pd.Categorical(
        avg_metric_per_matchday['match_day'],
        categories=matchday_order,
        ordered=True
    )
    avg_metric_per_matchday = avg_metric_per_matchday.sort_values(by='match_day')
    avg_metric_per_matchday_plot = avg_metric_per_matchday.copy() 
    avg_metric_per_matchday.rename(columns={'match_day': 'Match Day',
                                            'distance_km':'Distance',
                                            'sprint_distance_m':'Sprint Distance',
                                            'power_plays':'Power Plays',
                                            'energy_kcal':'Energy',
                                            'impacts':'Impacts',
                                            'total_accelerations':'Accelerations',
                                            'total_decelerations': 'Decelerations',
                                            'total_actions':'Total Actions',
                                            'player_load':'Player Load',
                                            'top_speed_kmh':'Top Speed',
                                            'distance_per_min_mmin':'Distance per Minute',
                                            'power_score_wkg':'Power Score',
                                            'work_ratio':'Work Ratio',
                                            'max_acceleration_mss':'Max Acceleration',
                                            'max_deceleration_mss':'Max Deceleration',
                                            'acc_counts_per_min':'Acceleration Counts per Min',
                                            'dec_counts_per_min':'Deceleration Counts per Min'

                                            }, inplace=True)
    avg_metric_per_matchday.drop(columns=['Max Acceleration','Max Deceleration'], inplace=True) 
    styled = style_table_for_docs(avg_metric_per_matchday)
    return styled, avg_metric_per_matchday.round(1), avg_metric_per_matchday_plot.round(2)

In [ ]:
def plot_club_metrics_across_matchdays(club_df, volume_metrics, intensity_metrics, matchday_order):
    avg_metric_per_matchday = get_avg_metric_per_matchday(club_df, volume_metrics, intensity_metrics, matchday_order, style_table_for_docs)[2]
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    metrics = [
        ('distance_km', 'Distance (km)'),
        ('sprint_distance_m', 'Sprint Distance (m)'),
        ('player_load', 'Player Load'),
        ('top_speed_kmh', 'Top Speed (km/h)')
    ]
    md15_idx = matchday_order.index('Md11')
    for ax, (col, label) in zip(axes.flatten(), metrics):
        sns.lineplot(
            data=avg_metric_per_matchday,
            x='match_day',
            y=col,
            marker='o',
            ax=ax,
            label=f'Average {label}'
        )
        avg_value = avg_metric_per_matchday[col].mean()
        ax.axhline(avg_value, color='red', linestyle='--', label=f'Season Average: {avg_value:.2f}')
        ax.axvline(md15_idx, color='black', linestyle=':', linewidth=1)
        ax.axvspan(-0.5, md15_idx - 0.5, color='skyblue', alpha=0.35)
        ax.axvspan(md15_idx - 0.5, md15_idx + 0.5, color='lightgreen', alpha=0.35)
        ax.axvspan(md15_idx + 0.5, len(matchday_order) - 0.5, color='lightgreen', alpha=0.35)
        xticklabels = [str(md).replace('Md', 'MD ') for md in avg_metric_per_matchday['match_day']]
        ax.set_xticks(range(len(avg_metric_per_matchday['match_day'])))
        ax.set_xticklabels(xticklabels, rotation=90)
        ax.set_title(f'{label} Across Match Days')
        ax.set_xlabel('Match Day')
        ax.set_ylabel(label)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.legend(frameon=False)
        ymin, ymax = ax.get_ylim()
        ax.text(md15_idx/2, avg_value - (ymax-ymin)*0.12, 'First Round', color='blue', fontsize=12, ha='center', va='top', alpha=0.7)
        ax.text(md15_idx + (len(matchday_order)-md15_idx)/2, avg_value + (ymax-ymin)*0.08, 'Second Round', color='green', fontsize=12, ha='center', va='bottom', alpha=0.7)
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()
    return fig


In [ ]:
def plot_speed_zone_pct_by_position(club_df, position_order=None, ax=None):
    speed_zone_cols = [
        'distance_in_speed_zone_1_km',
        'distance_in_speed_zone_2_km',
        'distance_in_speed_zone_3_km',
        'distance_in_speed_zone_4_km',
        'distance_in_speed_zone_5_km'
    ]
    # Group by position and sum distances (in km)
    zone_by_position = club_df.groupby('general_position')[speed_zone_cols].sum()
    # Reorder positions if desired
    if position_order is not None:
        zone_by_position = zone_by_position.reindex(position_order)
        zone_by_position.index = [f"{pos}s" for pos in position_order]
    else:
        zone_by_position.index = [f"{pos}s" for pos in zone_by_position.index]
    # Prepare labels
    zone_labels = [f"Zone {i+1}" for i in range(len(speed_zone_cols))]
    zone_by_position.columns = zone_labels
    # Convert to percentages row-wise
    zone_pct = zone_by_position.div(zone_by_position.sum(axis=1), axis=0) * 100
    # Plot
    if ax is None:
        ax = zone_pct.plot(
            kind='bar',
            stacked=True,
            figsize=(10, 6),
            color=sns.color_palette('viridis', n_colors=len(zone_labels))
        )
    else:
        zone_pct.plot(
            kind='bar',
            stacked=True,
            ax=ax,
            color=sns.color_palette('viridis', n_colors=len(zone_labels))
        )
    # Annotate values inside each bar segment (as percentages, one decimal)
    for i, pos in enumerate(zone_pct.index):
        cumulative = 0
        for j, zone in enumerate(zone_labels):
            value = zone_pct.loc[pos, zone]
            height = value
            y = cumulative + height / 2
            if value > 0.05:
                ax.text(
                    i, y, f"{value:.1f}%",
                    ha='center', va='center', fontsize=9,
                    color='black' if j > 1 else 'white', fontweight='bold'
                )
            cumulative += height
    ax.set_xlabel('Position Group')
    ax.set_title('Percentage of Total Distance Covered by Position in Each Speed Zone', pad=30)
    ax.set_xticks(range(len(zone_pct.index)))
    ax.set_xticklabels(zone_pct.index, rotation=0)
    ax.legend(title='Speed Zone', frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.spines['top'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_color('#E0DCDD')
    ax.set_facecolor('#F5F5F5')
    ax.set_yticklabels([])
    ax.set_yticks([])
    ax.set_yticks(range(0, 101, 20))
    ax.set_ylim(0, 100)
    plt.tight_layout()
    return ax, zone_by_position, zone_pct


## Club Comparisons

In [ ]:
def club_vs_season_metric_comparison(club_df, df, volume_metrics,intensity_metrics, club, style_table_for_docs):
  metrics = volume_metrics + intensity_metrics
  # 1. Calculate club average
  club_avg = club_df[metrics].mean().to_frame(name='Club Average')
  # 2. Calculate season average
  season_avg = df[metrics].mean().to_frame(name='Season Average')
  # 3. Combine them
  comparison_df = club_avg.join(season_avg)
  
  # 5. Final combined table
  
  comparison_df = comparison_df.round(2).T
  comparison_df.rename(columns={'index': 'Metric',
                                'distance_km':'Distance',
                                            'sprint_distance_m':'Sprint Distance',
                                            'power_plays':'Power Plays',
                                            'energy_kcal':'Energy',
                                            'impacts':'Impacts',
                                            'total_accelerations':'Accelerations',
                                            'total_decelerations': 'Decelerations',
                                            'total_actions':'Total Actions',
                                            'player_load':'Player Load',
                                            'top_speed_kmh':'Top Speed',
                                            'distance_per_min_mmin':'Distance per Minute',
                                            'power_score_wkg':'Power Score',
                                            'work_ratio':'Work Ratio',
                                            'max_acceleration_mss':'Max Acceleration',
                                            'max_deceleration_mss':'Max Deceleration',
                                            'acc_counts_per_min':'Acceleration Counts per Min',
                                            'dec_counts_per_min':'Deceleration Counts per Min'}, inplace=True)
  comparison_df.drop(columns=['Max Acceleration','Max Deceleration'], inplace=True) 

  #6. calculate percentage
  comparison_df= comparison_df.T
  comparison_df['% Difference'] = ((comparison_df['Club Average'] - comparison_df['Season Average']) / comparison_df['Season Average']) * 100

  # 7. Reorder columns
  ordered_cols = ['Club Average', 'Season Average', '% Difference']
  comparison_df = comparison_df[ordered_cols]

  comparison_df= comparison_df.T

  # 8. Display styled
  return style_table_for_docs(comparison_df), comparison_df.round(2)


In [ ]:
def positional_metric_comparison(club_df, df, volume_metrics, intensity_metrics, style_table_for_docs, club):
    metrics = volume_metrics + intensity_metrics
    club_avg = club_df.groupby('general_position')[metrics].mean().add_prefix("Club_")
    season_avg = df.groupby('general_position')[metrics].mean().add_prefix("Season_")
    combined = club_avg.join(season_avg)
    for metric in metrics:
        club_col = f"Club_{metric}"
        season_col = f"Season_{metric}"
        combined[f"Percentage_Difference_{metric}"] = (
            (combined[club_col] - combined[season_col]) / combined[season_col]
        ) * 100
    ordered_cols = []
    for metric in metrics:
        ordered_cols.extend([
            f"Club_{metric}",
            f"Season_{metric}",
            f"Percentage_Difference_{metric}"
        ])
    comparison_by_position = combined[ordered_cols].round(2).T
    comparison_by_position.reset_index(inplace=True)
    comparison_by_position.rename(columns={'index': 'Metric'}, inplace=True)
    
    styled = style_table_for_docs(comparison_by_position).set_caption(f"{club} vs League Positional Metrics Comparison")
    return styled, comparison_by_position


In [ ]:
import os
import math
from docx import Document
from docx.shared import Inches

# Helper to format cell values safely
def fmt(val, float_fmt="{:.2f}"):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return ""
    if isinstance(val, (int,)):
        return str(val)
    if isinstance(val, float):
        return float_fmt.format(val)
    return str(val)

def add_df_to_docx_table(doc, df):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = 'Table Grid'
    # Add header
    hdr_cells = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr_cells[i].text = str(col)
    # Add rows
    for idx, row in df.iterrows():
        row_cells = table.add_row().cells
        for i, val in enumerate(row):
            row_cells[i].text = str(val) if pd.notnull(val) else ""
    return table

out_dir = "reports_wsl"
os.makedirs(out_dir, exist_ok=True)

doc = Document()
doc.add_heading(f'{club} Catapult Report', 0)
doc.add_page_break()  # Add a page break for better organization


# Add introduction section
doc.add_heading('Introduction', level=1)
doc.add_paragraph(
    '''In a strategic move to elevate domestic football standards, the Federation of Uganda Football Associations (FUFA) has invested substantially in Catapult GPS systems for clubs in the Uganda Premier League and FUFA Women Super League. By equipping teams with this state-of-the-art performance monitoring technology, FUFA demonstrates its commitment to player development, optimized match preparation, and sustained competitive excellence.
On September 13 2024, The FUFA President, Hon. Eng Moses Hashim Magogo said “The data collected from the devices will not only benefit the individual clubs but will also be shared with the national team coaches” He further went on to say “This will enable the coaches to select players based on their performance data and tailor training sessions to meet the specific needs of the national team”
The FUFA Research, Science and Technology (RST) has prepared this report focusing on BUL FC’s physical performance during the 2024/25 season, leveraging Catapult tracking data to deliver actionable insights. The major objectives are to:
•	Quantify player workloads and intensity metrics
•	Identify performance trends and outliers
•	Inform training load management, injury prevention, and tactical planning
It is important to note that accurate analysis and subsequent interpretation depend on consistent device usage, precise session tagging, and timely data synchronization. Where gaps arise due to technical or logistical constraints, we highlight their impact on the analyses.
The report is organized into seven sections:
•	Methodology: Data collection, cleaning, and analysis protocols
•	Key Concepts and Definitions in Physical Performance Analysis
•	Season Results: Individual and aggregated performance metrics (e.g., total distance, high-intensity efforts, accelerations)
•	Club Comparison: BUL FC versus league averages to reveal relative strengths and areas for improvement
•	Challenges: Recurring issues such as inconsistent device usage, mislabelled sessions, failure to upload, mismanagement of equipment
•	Recommendations: Corrective measures including refresher training for Catapult operators, compliance to session naming protocols
•	FUFA Future Plans: Strategic initiatives to enhance data integration, reporting consistency, and long-term performance monitoring across all top-tier clubs
''')

doc.add_heading('Methodology', level=1)

doc.add_paragraph('''
This report is based on data collected using the Catapult GPS player tracking system during the 2024/25 football season. The data includes performance metrics such as total distance covered, sprint distance, work ratio, player load, session availability, top speed, accelerations, decelerations, energy, power plays, impacts, total actions, distance per minute.

Data was collected and uploaded by trained club staff immediately after match sessions. The FUFA Research, Science & Technology (RST) unit oversaw the data collection process, ensuring sessions were split by halves, correctly labelled, and uploaded within seventy-two (72) hours post-match. Where necessary, additional follow-up was done with the club operators to correct mislabelled sessions or fill in missing data, ensuring full coverage of match data

The rigorous data cleaning process involved removing duplicates, resolving inconsistencies in player-pod assignments, and verifying that each match session met the minimum completeness threshold (e.g., full ninety (90) minutes, sufficient number of tracked players). Any session failing these criteria was excluded from further analysis to maintain consistency.

Once cleaned, data was aggregated by individual player and by club. Metrics were normalized to account for variations in match frequency, squad size, and overall data availability, and clubs with fewer than ten complete sessions were omitted from comparative analyses. All statistical summaries and visualizations were generated using Microsoft Excel, Power BI, R, and Python, following established RST workflows to ensure reliable, consistent benchmarks for technical review and long-term planning.
''')

doc.add_page_break()  # Add a page break for better organization

doc.add_heading('Key Concepts and Definitions in Physical Performance Analysis', level=1)

doc.add_paragraph('''This section gives a basic introduction to coaches, analysts and any other personnel who may be interested in physical performance data. It includes basic definitions and brief descriptions of the metrics and terminology used in the field. It also includes a section of why a club should be interested in their physical data.
Physical performance analysis: A method of collecting and interpreting data on a player’s movement and workload during training or matches. By measuring total and sprint distance, player load and work ratios, performance staff can monitor fitness adaptations, manage fatigue and implement injury-prevention protocols. In the Uganda Premier League, several clubs have used these insights to adjust training loads and optimise readiness for crucial fixtures.
Catapult GPS technology: A wearable athlete-monitoring system that integrates global positioning sensors with inertial measurement units (accelerometers, gyroscopes and magnetometers) to capture real-time movement data at high sampling rates. After each session, devices sync to base stations and upload into the Catapult software suite, which calculates metrics such as sprint profiles, accelerations and overall player load. Clubs use this information to tailor individual conditioning programs and track recovery between matchdays.
Volume: This refers to the total amount of physical work performed by a player over a training session or match. It captures aggregated outputs such as total distance covered or total number of accelerations and decelerations. Volume metrics help coaches gauge overall workload across a full session. Volume metrics include total distance, sprint distance, power plays, energy, impacts, accelerations, decelerations, total actions and player load.
Intensity: This refers to the rate or magnitude at which physical work is carried out. It measures how quickly or forcefully a player executes movements, for example distance per minute or accelerations per minute. Clubs use intensity metrics to adjust training drills for peak performance on matchdays. Intensity metrics include top speed, distance per minute, work ratio, power score, acceleration counts per minute and deceleration counts per minute.



Catapult Metrics and Their Meaning
Metric	Definition	Unit	Volume or Intensity
Total Distance	The total number of kilometres a player covers while walking or running in a session.	Kilometres(km)	Volume
Sprint Distance	Distance covered while running above a set speed threshold of 25.2 km/h.	Metres(m)	Volume
Power Plays	The number of intense actions a player was involved in.	Count	Volume
Energy	A measure of how much energy is expended during a session.	Kilocalories(kcal)	Volume
Impacts	The number of high-force events (e.g., collisions, landings, tackles	Count	Volume
Accelerations	Total count of acceleration events above a set threshold	Count	Volume
Decelerations	Total count of acceleration events above a set threshold	Count	Volume
Total Actions	Sum of total accelerations and total decelerations	Count	Volume
Player Load	Measures the overall workload on a player’s body in a session.	Arbitrary Units (AU)	Volume
Top Speed	The maximum speed a player reaches in a session.	Kilometres per Hour (km/h)	Intensity
Distance per Minute	Intensity measure showing meters covered every minute.	Metres per Minute (m/min)	Intensity
Work Ratio	Ratio of high-intensity work time to total time	Percentage	Intensity
Power Score	Measures the power output used per kilogram of a player’s weight.	Watts per Kilogram (w/kg)	Intensity
Acceleration Counts Per Minute	Total accelerations divided by minutes played	counts per minute	Intensity
Deceleration Counts Per Minute	Total accelerations divided by minutes played	counts per minute	Intensity
                  
Why should a club be interested in these metrics?
1.	Monitoring these metrics allows coaches to identify fatigue or underperformance before it leads to injury or decline in form. For example, a sudden drop in total actions or distance per minute can signal that a player is not recovering adequately between sessions.
2.	These measures enable position-specific benchmarks, ensuring defenders, wingers, midfielders and strikers train to the demands of their roles rather than a generic team standard. Comparing work ratio and power score across similar positions helps staff tailor workloads and maintain optimal performance levels.
3.	Individualized training programs are designed using these data points. If a player’s accelerations per minute fall below the team standard, practice drills can be adjusted to include more high-intensity runs and recovery phases, thereby bridging the gap to match demands.
4.	Objective feedback derived from metrics like player load and total impacts supports both performance reviews and recovery protocols. Quantitative insights align player perceptions of exertion with actual workload, guiding nutrition, rest and physiotherapy interventions for efficient return to peak readiness.
''')

doc.add_page_break()  # Add a page break for better organization

doc.add_paragraph('''
Speed Zones
There are five speed zones in Catapult. For the 2024/25 season, these were the zones and thresholds that were used. Note that these zones will be regularly reviewed by the RST team and adjusted at the end of the season to promote steady improvement.
Zone	Threshold (km/h)	Intensity	Descriptor
Zone 1	0 – 3.6	Very Low	Standing, Walking at Recovery Pace
Zone 2	3.61 – 10.8	Low to Moderate	Jogging and Easy Runing
Zone 3	10.81 – 18	Moderate to High	Running
Zone 4	18.1 – 25.2	High	High Speed Running
Zone 5	25.21 - 43.2	Maximal	Sprinting
''')

# -------- 1. Summary of Player Metrics --------
doc.add_heading('2025/2025 Season Results', level=1)
doc.add_heading('Match Day Usage', level=2)

doc.add_paragraph(f"Number of Matches Analysed: {club_df['match_day'].nunique()}")
# doc.add_paragraph(f"Number of Matches Uploaded: {uploaded_matches.get(club,1)}")
# doc.add_paragraph(f"Percentage of Matches Uploaded: {((club_df['match_day'].nunique()/uploaded_matches.get(club))*100)}%")


styled_match_stats, _ = get_matchday_stats_styled(club_df, matchday_order, style_table_for_docs, club)
add_df_to_docx_table(doc, styled_match_stats.data)

doc.add_paragraph()  # spacing
doc.add_heading('Player Usage', level=1)
styled_players, _ = get_players_monitored_stats(club_df, style_table_for_docs, club)
add_df_to_docx_table(doc, styled_players.data)

doc.add_paragraph()

# Insert styled metric summary table
doc.add_heading('Club Metric Results', level=2)
styled_metric_summary, _ = get_metric_summary_styled(club_df, volume_metrics, intensity_metrics, club, style_table_for_docs)
metric_summary_html = styled_metric_summary.to_html()
add_df_to_docx_table(doc, styled_metric_summary.data)

doc.add_paragraph()  # spacing

# -------- 2. Top Players by Metric --------
doc.add_heading('Top Player by Metric', level=1)
_,styled_top_players = get_top_players_by_metric(club_df, volume_metrics + intensity_metrics, style_table_for_docs)
add_df_to_docx_table(doc, styled_top_players)

doc.add_paragraph()


# -------- 5. Metrics Per Position  --------
doc.add_heading('Average Metrics Per Position', level=1)
_,styled_avg_metrics_by_position = get_avg_metrics_by_position(club_df, volume_metrics, intensity_metrics, style_table_for_docs)
add_df_to_docx_table(doc, styled_avg_metrics_by_position)

doc.add_paragraph()

# -------- 6. Metric Per Matchday --------
doc.add_heading('Average Metrics Per Matchday', level=1)
styled_avg_metrics_per_match = get_avg_metric_per_matchday(club_df, volume_metrics, intensity_metrics, matchday_order,style_table_for_docs)
add_df_to_docx_table(doc, styled_avg_metrics_per_match[1])

doc.add_paragraph()

# -------- 8. Comparison  --------
doc.add_heading('Club Comparison', level=1)
doc.add_heading('Comparison Against Season Averages', level=2)
_,styled_comparison= club_vs_season_metric_comparison(club_df, df,volume_metrics,intensity_metrics,club, style_table_for_docs)
add_df_to_docx_table(doc, styled_comparison.T.reset_index())

doc.add_paragraph()

# -------- 9. Comparison by Position --------
doc.add_heading('Comparison by Position', level=2)
styled_position_comparison, _ = positional_metric_comparison(club_df, df,volume_metrics,intensity_metrics, style_table_for_docs, club)
add_df_to_docx_table(doc, styled_position_comparison.data)

doc.add_paragraph()

# -------- 6. Export plot figure and add to document --------
doc.add_heading('Club Metrics Trend', level=1)
from io import BytesIO

img_stream = BytesIO()
fig = plot_club_metrics_across_matchdays(club_df, volume_metrics, intensity_metrics, matchday_order)
fig.savefig(img_stream, format='png', bbox_inches='tight')
img_stream.seek(0)
doc.add_picture(img_stream, width=Inches(6))

doc.add_heading('Positional Comparison in Each Speed Zone', level=1)
img_zone_stream = BytesIO()
ax, zone_by_position, zone_pct= plot_speed_zone_pct_by_position(club_df, position_order=None)
ax.figure.savefig(img_zone_stream, format='png', bbox_inches='tight')
img_zone_stream.seek(0)
doc.add_picture(img_zone_stream, width=Inches(6))

doc.add_page_break()  # Add a page break for better organization

doc.add_heading('Challenges', level=1)
doc.add_paragraph('''The challenges outlined below represent common operational and data management issues observed across multiple clubs using Catapult systems. These are not specific to any one club, but rather reflect broader trends that can impact the consistency, accuracy, and usefulness of performance data. From equipment mismanagement to gaps in staff training and session documentation, these hurdles can hinder the full potential of Catapult insights. Clubs experiencing similar difficulties or seeking tailored support are encouraged to contact the FUFA Research, Science and Technology (RST) Team, who can provide club-specific guidance and help implement effective solutions.
The challenges are outlined below:
1.	Inconsistent use of Catapult devices across games, leading to incomplete match coverage
2.	Failure to upload sessions at all or delays in syncing which creates backlogs
3.	Misclassification of training sessions as games, and vice versa
4.	Incorrect session naming, making it difficult to locate or aggregate sessions
5.	Improper session splitting (e.g., durations recorded as < 90 minutes or > 120 minutes, incorrect splits for substituted players)
6.	Mismanagement of equipment leading to loss or damage
7.	Limited expertise in translating raw Catapult outputs into tactical or physiological insights
8.	Club staff not fully trained on usage and upload procedures, resulting in missed or late uploads
''')
doc.add_page_break()  # Add a page break for better organization
doc.add_heading('FUFA Future Plans', level=1)
doc.add_paragraph('''This section outlines the strategic initiatives FUFA intends to roll out to maximize the impact of Catapult data across Uganda’s top tier leagues. These federation-led actions build on the 2024/25 season insights and are designed to standardize workflows, deepen analysis, and support both club staff and players. For club-specific implementation advice or technical assistance, please contact the FUFA Research, Science and Technology (RST) unit.
1.	Conduct regular visits to the clubs to monitor equipment usage and share club specific insights.
2.	Organise regular training workshops for club performance staff on Catapult device management, data procedures and software analytics to ensure consistent, high-quality data capture.
3.	Leverage catapult data to inform nutritional guidance, tailoring macronutrient ratios to individual training demands and recovery needs.
4.	Introduce individual performance benchmarks for every playing position (right back, centre forward, defensive midfielder, etc.) with detailed threshold bands for both volume and intensity at club level.
5.	Integrate Catapult metrics with tactical and technical analysis by linking GPS outputs to technical or tactical data generated by other systems, thereby providing coaches with combined insights into player movement patterns and team strategy.
6.	Carry out Research projects using catapult data along other data
''')
doc.add_page_break()  # Add a page break for better organization
doc.add_heading('Conclusion', level=1)
doc.add_paragraph('''We extend our sincere thanks to every coach, performance staff member and analyst for their dedication in collecting and interpreting Catapult data this season. Your hard work has laid a solid foundation for evidence-based decision-making across fitness, recovery and match preparation.
All Catapult outputs and associated player information will be handled in strict accordance with the FUFA data privacy and security policy, ensuring confidentiality, ethical use and compliant storage. This protocol safeguards both individual rights and the integrity of our performance analysis.
As we move into the next season, we look forward to your feedback and stand ready to support your club through FUFA’s Research, Science and Technology unit. You can reach us via email on  fufa.rst@gmail.com
''')

# -------- Save the document --------
out_file = os.path.join(out_dir, f"{club}_analysis_report.docx")
doc.save(out_file)
print(f"Saved report to: {out_file}")
